# M7 · LoRA with real tools (MLX & PEFT)

In M6 you built LoRA by hand: a frozen layer, plus a small update `scale · B @ A` that trains. It tuned Qwen2.5-0.5B with a fifth of one percent of its weights.

Nobody writes that layer by hand for real work. They use a library that does it, and does the parts M6 skipped: saving the adapter, loading it back, and training fast on bigger models. This module uses the two you'll meet most:

- **mlx-lm**, Apple's library for language models on the Mac's GPU
- **PEFT**, Hugging Face's library for PyTorch

You'll train the same M5 task on a model three times bigger, Qwen2.5-1.5B-Instruct, with both, and see that each tool is doing exactly what you built.

By the end of this lesson you'll have:

- written the M5 data in the chat format the tools read
- set up the same rank-8 adapter in mlx-lm and in PEFT, and found your M6 A, B and scale inside each
- trained a short run with each on the same batches, and compared their speed and memory
- measured what padding does to tokens per second, and what sequence length does to memory
- saved an adapter, loaded it back, and checked it gives the same answers

**How to work through it** (same as M1 to M6)

- Run each cell from top to bottom with **Shift-Enter**, and read the **What each line does** box under it.
- When a cell says **Predict**, write your guess down before running it.
- New words are **bold** where they're explained, and collected in the glossary in `README.md`.
- Save with **Cmd-S** now and then so the dashboard sees your progress.

The first time, section 3 downloads Qwen2.5-1.5B-Instruct (about 3 GB). Both tools use that one copy. The lesson holds both tools' models at once and peaks at about 16 GB, so close other big apps first.

## 0. Why use a tool?

Here's what your M6 run did, if you've run it (otherwise, the reference run's numbers).

In [ ]:
import json
from pathlib import Path

m6 = Path("../../outputs/m06/run.json")
if m6.exists():
    s = json.loads(m6.read_text())
    print(f"Your M6 run: held-out loss {s['heldout_before']:.3f} -> {s['heldout_after']:.3f}")
    print(f"  trained {s['trainable_pct']}% of the weights, adapter {s['adapter_mb']} MB")
else:
    print("Reference M6 run: held-out loss 1.554 -> 0.903, 0.22% of the weights, adapter 4.4 MB")

**What each line does**

- `Path("../../outputs/m06/run.json")`: the summary your M6 `train.py` wrote. The notebook runs from `modules/m07_lora_tools/`, so `../../` is the course folder.
- `.exists()`: `True` if the file is there. If you haven't run M6, the reference numbers print instead.
- `json.loads(m6.read_text())`: reads the file and turns its JSON into a Python dict (M5).

It worked. But look at what M6 left for you to do by hand: walk the model to find the layers, swap them, freeze the rest, save A and B in a format only your code understands, and remember the rank and scale so you could ever load them again.

A library does all of that, tested, for every common model. It also saves the adapter in a standard format, with its settings next to it, so anyone with the same base model can load it. And it's fast enough for bigger models. This module moves up from 0.5 to 1.5 billion parameters.

## 1. Two tools

**MLX** is Apple's array library, like PyTorch but built for Apple-silicon chips. Its arrays live in the Mac's unified memory, which the CPU and GPU share, so nothing is copied between them. **mlx-lm** is the library on top of it for language models: loading them, generating, and fine-tuning with LoRA. It runs only on Apple-silicon Macs.

**PEFT** (parameter-efficient fine-tuning) is Hugging Face's library of ways to tune a model without changing every weight, LoRA among them. It wraps the PyTorch models you used in M5 and M6, so it runs anywhere PyTorch does: here on the Mac's GPU (`mps`), elsewhere on NVIDIA GPUs.

Both are installed with the course's packages. Here are their versions, and where each will run.

In [ ]:
import mlx.core as mx
import mlx_lm
import peft
import torch

print("mlx-lm", mlx_lm.__version__, "on", mx.default_device())
print("peft", peft.__version__, "with torch", torch.__version__)
print("PyTorch can use the Mac's GPU:", torch.backends.mps.is_available())

**What each line does**

- `import mlx.core as mx`: MLX's arrays and maths, by convention called `mx` (like `np` for NumPy).
- `mx.default_device()`: where MLX computes: `Device(gpu, 0)`, the Mac's GPU.
- `torch.backends.mps.is_available()`: whether PyTorch can use the GPU through Apple's Metal (M1).

## 2. The data format

Both tools read training examples as a **chat record**: a list of messages, each a dict with a `role` (`system`, `user` or `assistant`) and its `content`. It's the same structure M3's chat template turned into tokens.

Files of examples are usually **JSONL**: one JSON object per line, so a file can be read one example at a time. mlx-lm's command line reads a folder with `train.jsonl` and `valid.jsonl` in exactly this format (`train.py` writes them for you in `outputs/m07/data/`).

Here's one M5 pair as a chat record.

In [ ]:
DATA = Path("../m05_full_finetune/data")
SYSTEM = "Rewrite the user's text in plain English. Reply with the rewrite only."
rows = [json.loads(line) for line in (DATA / "train.jsonl").read_text().splitlines()]

pair = rows[0]
record = {
    "messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": pair["jargon"]},
        {"role": "assistant", "content": pair["plain"]},
    ]
}
print(json.dumps(record))

**What each line does**

- `DATA = Path("../m05_full_finetune/data")`: M5's data, read in place.
- `[json.loads(line) for line in ...splitlines()]`: one dict per line: that's reading JSONL.
- `record = {"messages": [...]}`: the system prompt, the user's jargon and the assistant's plain rewrite, in the order they'd be said. This is exercise 1.
- `json.dumps(record)`: the dict as one line of JSON text: one line of a JSONL file.

The tokenizer turns a chat record into the token ids the model reads. Qwen2.5-1.5B has the same tokenizer and chat template as the 0.5B model from M5.

In [ ]:
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
text = tok.apply_chat_template(record["messages"], tokenize=False)
print(text)
prompt = record["messages"][:2]
prompt_ids = tok.apply_chat_template(prompt, add_generation_prompt=True)["input_ids"]
full_ids = tok.apply_chat_template(record["messages"])["input_ids"]
print(len(prompt_ids), "prompt tokens,", len(full_ids) - len(prompt_ids), "answer tokens")

**What each line does**

- `AutoTokenizer.from_pretrained(MODEL)`: the 1.5B model's tokenizer (M3). This downloads the model's small files the first time.
- `apply_chat_template(..., tokenize=False)`: the record as the text the model sees, with Qwen's `<|im_start|>` and `<|im_end|>` markers (M3 section 6).
- `prompt = record["messages"][:2]` with `add_generation_prompt=True`: just the prompt, ending where the assistant starts.
- The difference in length is the answer: the only tokens the loss counts (M5).

## 3. LoRA in mlx-lm

First, load the model in mlx-lm. `load` downloads it the first time (about 3 GB) and returns the model and a tokenizer.

In [ ]:
from mlx_lm import load

mlx_model, mlx_tok = load(MODEL)
attn = mlx_model.layers[0].self_attn
print(attn)
print("q_proj weight:", attn.q_proj.weight.shape, attn.q_proj.weight.dtype)
print(len(mlx_model.layers), "blocks")

**What each line does**

- `load(MODEL)`: reads the model from the Hugging Face cache (downloading it once) and converts it to MLX arrays.
- `mlx_model.layers[0].self_attn`: the attention part of the first block, with the same `q_proj`, `k_proj`, `v_proj` and `o_proj` you wrapped in M6.
- `.weight.dtype` is `bfloat16`: the model is stored in **bf16**, a 2-byte number format with float32's range but less precision. Both tools train in bf16 here, half the memory of M5's float32.

mlx-lm describes an adapter with a small dict, its **adapter config**. The part about LoRA is `lora_parameters`:

- `rank`: your `r`
- `scale`: the number `B @ A` is multiplied by. mlx-lm wants the scale itself, not alpha: you work out `alpha / r`
- `dropout`: 0, as in M6
- `keys`: which layers to wrap. mlx-lm names them from inside a block, so `q_proj` is `self_attn.q_proj`. These are its **keys** (what PEFT and M6 call target modules)

Next to it, `num_layers` says how many blocks get adapters, counted from the top. All 28 here.

**Predict:** the 1.5B model's `q_proj` and `o_proj` are 1536 by 1536; `k_proj` and `v_proj` are 1536 inputs by 256 outputs. How many numbers does rank 8 on all four, in all 28 blocks, train? (M6's `lora_params` counts one layer.)

In [ ]:
from mlx.utils import tree_flatten
from mlx_lm.tuner.utils import linear_to_lora_layers

r, alpha = 8, 16
lora = {
    "rank": r,
    "scale": alpha / r,
    "dropout": 0.0,
    "keys": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj"],
}
mlx_model.freeze()
linear_to_lora_layers(mlx_model, 28, lora)
trainable = sum(v.size for _, v in tree_flatten(mlx_model.trainable_parameters()))
total = sum(v.size for _, v in tree_flatten(mlx_model.parameters()))
print(f"{trainable:,} of {total:,} train ({100 * trainable / total:.3f}%)")
print(mlx_model.layers[0].self_attn.q_proj)

**What each line does**

- `lora = {...}`: the adapter config. `"scale": alpha / r` is 2.0, the same scale as your M6 layer. This is exercise 2.
- `mlx_model.freeze()`: freezes every weight (M6's `requires_grad_(False)` on everything).
- `linear_to_lora_layers(mlx_model, 28, lora)`: mlx-lm's version of your `inject_lora`: wraps the `keys` layers in the top 28 blocks.
- `mlx_model.trainable_parameters()`: just the parameters that aren't frozen: the adapters. `tree_flatten` lists them as (name, array) pairs so we can add up their sizes.
- The last line prints a wrapped layer: it's now a `LoRALinear`, the same name as your M6 class.

28 blocks × 8 × (3072 + 1792 + 1792 + 3072) = 2,179,072: 0.141% of the model. That's a smaller share than M6's 0.22%: the model is three times bigger, but its layers are only about 1.7 times wider, and the adapter grows only with width.

Now look inside the wrapped layer. Here's mlx-lm's own `__call__` (its `forward`).

In [ ]:
import inspect

from mlx_lm.tuner.lora import LoRALinear

print(inspect.getsource(LoRALinear.__call__))
q = mlx_model.layers[0].self_attn.q_proj
print("lora_a", q.lora_a.shape, " lora_b", q.lora_b.shape, " scale", q.scale)
print("lora_b is all zeros:", bool((q.lora_b == 0).all()))

**What each line does**

- `inspect.getsource(...)`: prints the source code of a function, here mlx-lm's LoRA forward pass.
- `lora_a` is 1536 x 8 and `lora_b` is 8 x 1536: your A and B, stored the other way round (transposed), so mlx-lm computes `x @ lora_a @ lora_b` without the `.T`s.
- `scale` is 2.0, and `lora_b` starts at zero, exactly as in your M6 layer: at the start it changes nothing.

Same idea, line for line: the frozen layer's output, plus `scale` times x through A and then B (mlx-lm adds dropout too, which is 0 here).

## 4. LoRA in PEFT

PEFT keeps its settings in a `LoraConfig`. Two differences from mlx-lm:

- It takes **alpha** itself (`lora_alpha`) and works out the scale on its own.
- Its `target_modules` are the plain layer names, like `q_proj`, matched anywhere in the model.

`get_peft_model` does what your `inject_lora` did: wraps the targets and freezes everything else.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("mps")
base.generation_config.repetition_penalty = 1.0  # decode the same way as mlx-lm (see below)
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM",
)
peft_model = get_peft_model(base, config)
peft_model.print_trainable_parameters()

**What each line does**

- `from_pretrained(MODEL, dtype=torch.bfloat16).to("mps")`: the same model, read from the same downloaded files, as a PyTorch model in bf16 on the GPU.
- `repetition_penalty = 1.0`: Qwen's settings ask Hugging Face to discourage repeated words when it generates, even for greedy replies; mlx-lm doesn't. Turning it off makes the two tools decode the same way.
- `LoraConfig(...)`: the same adapter: rank 8, alpha 16 (PEFT divides), the four attention projections, no dropout. `task_type="CAUSAL_LM"` tells it the model predicts the next token. This is exercise 3.
- `get_peft_model(base, config)`: wraps the target layers and freezes the rest.
- `print_trainable_parameters()`: PEFT's own count: the same 2,179,072, 0.141%.

In [ ]:
q = peft_model.base_model.model.model.layers[0].self_attn.q_proj
print(q)
print("A:", tuple(q.lora_A["default"].weight.shape), " B:", tuple(q.lora_B["default"].weight.shape))
print("scale:", q.scaling["default"])
print("A and B are stored in", q.lora_A["default"].weight.dtype)

**What each line does**

- `peft_model.base_model.model.model.layers[0]`: PEFT wraps the model twice (its own wrapper, then the LoRA model), so the path is long. The rest is the M5 path to a layer.
- `q.lora_A["default"]` and `q.lora_B["default"]`: A and B, as small `nn.Linear` layers. `"default"` is the adapter's name: PEFT can hold several adapters on one model.
- A is 8 x 1536 and B is 1536 x 8: exactly your M6 shapes.
- `q.scaling["default"]` is 2.0: 16 / 8, worked out by PEFT.
- PEFT keeps A and B in float32 even though the model is bf16, so small updates aren't rounded away.

So the two tools build the same adapter: same rank, same scale, same layers, same 2,179,072 numbers. Only the names and the storage differ.

## 5. A short run with each

To compare tools fairly, they must do the same work: the same examples, in the same batches, in the same order, with the same loss. So we build the batches once, in PyTorch, and hand mlx-lm the same token ids.

Each batch has `input_ids`, an `attention_mask` (1 for a real token, 0 for padding) and `labels` (the answer tokens, with -100 where the loss should ignore a position), as in M5. Every batch is also padded to a multiple of 32 tokens: mlx-lm compiles its training step for each batch shape, so fewer shapes means less compiling.

In [ ]:
import time

import torch.nn.functional as F

STEPS = 20


END = tok.convert_tokens_to_ids("<|im_end|>")


def example(row):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": row["jargon"]},
    ]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True)["input_ids"]
    answer = tok.encode(row["plain"], add_special_tokens=False) + [END]
    return prompt + answer, [-100] * len(prompt) + answer


def batch_of(rows):
    seqs = [example(r) for r in rows]
    longest = max(len(ids) for ids, _ in seqs)
    width = longest + (-longest % 32)
    ids = [s + [tok.pad_token_id] * (width - len(s)) for s, _ in seqs]
    mask = [[1] * len(s) + [0] * (width - len(s)) for s, _ in seqs]
    labels = [lab + [-100] * (width - len(lab)) for _, lab in seqs]
    return {
        "input_ids": torch.tensor(ids),
        "attention_mask": torch.tensor(mask),
        "labels": torch.tensor(labels),
    }


batches = [batch_of(rows[i * 8 : i * 8 + 8]) for i in range(STEPS)]
print(len(batches), "batches; the first is", tuple(batches[0]["input_ids"].shape))

**What each line does**

- `example(row)`: one example's token ids and labels: the prompt, then the answer ending in `<|im_end|>`; labels are -100 over the prompt (M5's `build_example`).
- `width = longest + (-longest % 32)`: rounds the longest sequence up to a multiple of 32. `-longest % 32` is how many tokens that takes.
- `batch_of(rows)`: pads every sequence to `width` (M5's `collate`), and returns the three tensors.
- `END`: the id of `<|im_end|>`, the token that ends every answer.
- `batches`: 20 batches of 8 examples: the first 160 training pairs, for a short run.

**mlx-lm.** Its trainer, `mlx_lm.tuner.trainer.train`, takes the model, an optimizer, and two things we pass in to keep the comparison fair: `iterate_batches`, which hands it our batches in order (by default it sorts examples by length), and `loss`, cross-entropy on the answer tokens only, the same loss as PyTorch's.

In [ ]:
import tempfile
import warnings

import mlx.nn as mnn
import mlx.optimizers as optim
from mlx_lm.tuner import trainer


def mlx_loss(model, ids, labels):
    logits = model(ids[:, :-1]).astype(mx.float32)
    targets = labels[:, 1:]
    keep = targets != -100
    ce = mnn.losses.cross_entropy(logits, mx.where(keep, targets, 0)) * keep
    return ce.sum() / keep.sum(), keep.sum()


mlx_batches = [(mx.array(b["input_ids"].numpy()), mx.array(b["labels"].numpy())) for b in batches]


def same_batches(**_):
    yield from mlx_batches


warnings.filterwarnings("ignore", message=".*ipywidgets")  # its progress bar, in Jupyter
mlx_dir = Path(tempfile.mkdtemp()) / "mlx_adapter"
mlx_dir.mkdir()
mx.reset_peak_memory()
start = time.perf_counter()
trainer.train(
    mlx_model,
    optim.AdamW(learning_rate=2e-4, weight_decay=0.0),
    train_dataset=mlx_batches,
    args=trainer.TrainingArgs(
        batch_size=8,
        iters=STEPS,
        steps_per_report=5,
        steps_per_eval=10**9,
        adapter_file=str(mlx_dir / "adapters.safetensors"),
    ),
    loss=mlx_loss,
    iterate_batches=same_batches,
)
mlx_seconds = time.perf_counter() - start
mlx_peak = mx.get_peak_memory() / 1e9
print(f"mlx-lm: {mlx_seconds:.1f}s, peak {mlx_peak:.1f} GB (MLX's own count)")

**What each line does**

- `mlx_loss(model, ids, labels)`: the model's predictions for each next token, cross-entropy against the labels, and `keep` to count only answer tokens (labels that aren't -100). It returns the loss and how many tokens it counted, which is what mlx-lm's trainer expects.
- `.astype(mx.float32)`: the loss is computed in float32, so bf16's rounding doesn't creep in.
- `mlx_batches`: the same token ids, as MLX arrays. `.numpy()` turns a PyTorch tensor into a NumPy array, which `mx.array` reads.
- `same_batches(**_)`: a generator that ignores mlx-lm's batching settings (`**_` swallows them) and yields our batches in order.
- `warnings.filterwarnings(...)`: hides a harmless note from the library mlx-lm draws its progress table with.
- `trainer.TrainingArgs(...)`: 20 steps, a report every 5, no evaluation (`steps_per_eval` far beyond the end), and where to save the adapter.
- `mx.get_peak_memory()`: the most memory MLX held at once since `mx.reset_peak_memory()`.

**PEFT.** PEFT doesn't have its own trainer: the model it returns is an ordinary PyTorch model, so we train it with the loop you wrote in M4 and M5, on the same batches.

In [ ]:
params = [p for p in peft_model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=2e-4, weight_decay=0.0)
peft_model.train()
torch.mps.synchronize()
start = time.perf_counter()
torch_peak = 0.0
for step, b in enumerate(batches):
    b = {k: v.to("mps") for k, v in b.items()}
    logits = peft_model(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).logits
    loss = F.cross_entropy(
        logits[:, :-1].float().reshape(-1, logits.shape[-1]),
        b["labels"][:, 1:].reshape(-1),
        ignore_index=-100,
    )
    opt.zero_grad()
    loss.backward()
    opt.step()
    torch_peak = max(torch_peak, torch.mps.driver_allocated_memory() / 1e9)
    if step % 5 == 4:
        print(f"  step {step + 1}  loss {loss.item():.3f}")
torch.mps.synchronize()
peft_seconds = time.perf_counter() - start
print(f"PEFT: {peft_seconds:.1f}s, peak {torch_peak:.1f} GB (PyTorch's own count)")
torch.mps.empty_cache()

**What each line does**

- `params`: only the parameters that train: the adapters. AdamW keeps notes for these alone (M6).
- `torch.mps.synchronize()`: waits for the GPU to finish. GPU work runs in the background, so without it the timer would stop too early.
- The loop is M5's: forward, the same masked cross-entropy, `zero_grad`, `backward`, `step`.
- `torch.mps.driver_allocated_memory()`: the memory PyTorch has taken from the GPU driver, including memory it has freed but kept for later.
- `torch.mps.empty_cache()`: hands that kept memory back, so the rest of the lesson has room.

Two different ways to count memory, from two different libraries, so the numbers above aren't comparable yet. mlx-lm counts what its arrays hold; PyTorch counts what it has taken from the driver, including memory it freed and kept to reuse. That's why `train.py` asks macOS instead: it reads each process's peak **footprint**, the number Activity Monitor shows, which counts GPU memory the same way for both. On the reference Mac, the full 100-step runs peaked at 9.3 GB (mlx-lm) and 12.8 GB (PEFT).

Speed is close. On the full runs, mlx-lm took 94 s and PEFT 86 s for the same 100 steps. Neither tool is simply "faster": it depends on the model, the batch shapes, and the versions you have.

## 6. Tokens per second

Seconds per run only compares runs that do the same work. To compare different batch sizes, sequence lengths or datasets, measure **tokens per second**: how many tokens training gets through each second.

But which tokens? A batch's shape counts **padding** too: the filler that makes every sequence in a batch the same length. The GPU processes padding like any other position, but the loss ignores it, so the model learns nothing from it.

**Predict:** in our 20 batches, what share of the positions do you think are padding?

In [ ]:
real = sum(int(b["attention_mask"].sum()) for b in batches)
positions = sum(b["attention_mask"].numel() for b in batches)
padding = 100 * (1 - real / positions)
print(f"{real:,} real tokens in {positions:,} positions: {padding:.0f}% padding")
for name, secs in [("mlx-lm", mlx_seconds), ("PEFT", peft_seconds)]:
    print(f"{name:7} {real / secs:6.0f} real tokens/s   {positions / secs:6.0f} with padding")

**What each line does**

- `int(b["attention_mask"].sum())`: the 1s in a batch's mask: its real tokens. This is exercise 4.
- `.numel()`: how many numbers a tensor holds: every position, padding included.
- The two speeds use the same seconds; only what's counted differs.

Padding makes a run look faster than it is. A run that pads more (longer, uneven batches) can show a higher padded speed while learning from fewer real tokens. So count real tokens: that's what `train.py`'s tokens/sec does, with your exercise 4. (It's also why mlx-lm's own trainer sorts examples by length before batching: similar lengths need less padding.)

## 7. Sequence length and memory

**Sequence length** is how many tokens one example has. Training keeps the **activations** of every layer for every token, so `.backward()` can use them (M5). Twice the tokens, twice the activations, and attention, which compares every token with every other, grows faster still.

**Predict:** one training step on 2 sequences of 64 tokens peaks at some amount of memory, the 3 GB of weights included. At 512 tokens, eight times as many, will the peak be about the same, about twice, or several times as much?

We measure one mlx-lm step at four lengths, with random token ids (the values don't matter for memory).

In [ ]:
loss_and_grad = mnn.value_and_grad(mlx_model, mlx_loss)
weights_gb = sum(v.nbytes for _, v in tree_flatten(mlx_model.parameters())) / 1e9
vocab = mlx_model.args.vocab_size
print(f"the model's weights alone: {weights_gb:.1f} GB; its vocabulary: {vocab:,} tokens")
for length in [64, 128, 256, 512]:
    ids = mx.random.randint(0, 1000, (2, length))
    mx.clear_cache()
    mx.reset_peak_memory()
    (loss, _), grads = loss_and_grad(mlx_model, ids, ids)
    mx.eval(loss, grads)
    peak = mx.get_peak_memory() / 1e9
    logits_gb = 2 * length * vocab * 4 / 1e9
    print(f"{length:4} tokens: peak {peak:5.1f} GB   (logits alone {logits_gb:.1f} GB)")

**What each line does**

- `mnn.value_and_grad(mlx_model, mlx_loss)`: MLX's way to get a loss and its gradients in one call (PyTorch's `loss.backward()`, as a function).
- `v.nbytes`: how many bytes an array takes. The weights are the floor: they're in memory at any length.
- `mlx_model.args.vocab_size`: how many tokens the model knows. For every position it outputs one score per token: the **logits** (M2).
- `mx.random.randint(0, 1000, (2, length))`: 2 random sequences of `length` tokens; using them as labels too makes every position count.
- `logits_gb`: 2 sequences x `length` positions x the vocabulary x 4 bytes (the loss works in float32): the size of just the logits.
- `mx.eval(loss, grads)`: MLX is lazy: it only computes when asked. This makes it do the step now, so the peak is real.

Above the 3.1 GB of weights, a step needed about 0.8 GB at 64 tokens, then 1.2, 2.5 and 5.0 GB. Past the shortest length it doubles each time the length doubles (at 64 tokens, fixed costs still show). The logits are part of it (0.6 GB at 512 tokens, because every position gets a score for each of 151,936 tokens), but most of it is the activations every layer keeps for the backward pass. At these lengths attention's extra growth is still small; at thousands of tokens it starts to show.

That's why `train.py` keeps `--max-len 160` (the longest M5 example is under that) and batches of 8. When memory runs out, shorter sequences or smaller batches are the first things to try; M8 adds another: gradient checkpointing.

## 8. Saving and loading an adapter

The point of an **adapter file** is that it's small and separate: the 3 GB base model stays as it is, and the few megabytes of A and B can be shared, swapped and loaded onto any copy of the same base.

mlx-lm saved its adapter at the end of training. It also needs the adapter config, so it knows how to wrap the layers before loading A and B into them.

In [ ]:
(mlx_dir / "adapter_config.json").write_text(
    json.dumps({"fine_tune_type": "lora", "num_layers": 28, "lora_parameters": lora})
)
peft_dir = Path(tempfile.mkdtemp()) / "peft_adapter"
peft_model.save_pretrained(peft_dir)
for folder in (mlx_dir, peft_dir):
    for f in sorted(folder.iterdir()):
        print(f"{f.parent.name}/{f.name:28} {f.stat().st_size / 1e6:8.3f} MB")

**What each line does**

- `adapter_config.json`: the `num_layers` and `lora_parameters` from section 3, as a file. This is what your exercise 2 returns.
- `peft_model.save_pretrained(peft_dir)`: PEFT saves only the adapter (A, B, and its LoraConfig as `adapter_config.json`), not the base model.
- `.stat().st_size`: a file's size in bytes.

Both adapter files are about 8.7 MB: 2,179,072 numbers, 4 bytes each (float32), against about 3,100 MB for the model. Ten adapters for ten tasks would still be smaller than one percent of the base.

Now load each back, on a fresh model, and check the replies match the model that trained. First mlx-lm: `load` takes `adapter_path`, reads the config, wraps the layers, and loads A and B.

In [ ]:
from mlx_lm import generate

question = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "We will leverage our core competencies to drive value."},
]
prompt = mlx_tok.apply_chat_template(question, add_generation_prompt=True)
trained_reply = generate(mlx_model, mlx_tok, prompt=prompt, max_tokens=60)
del mlx_model
mx.clear_cache()

reloaded, reloaded_tok = load(MODEL, adapter_path=str(mlx_dir))
reloaded_reply = generate(reloaded, reloaded_tok, prompt=prompt, max_tokens=60)
print("trained: ", trained_reply)
print("reloaded:", reloaded_reply)
print("same:", trained_reply == reloaded_reply)

**What each line does**

- `generate(model, tokenizer, prompt=..., max_tokens=60)`: mlx-lm's text generation; greedy by default.
- `del mlx_model` and `mx.clear_cache()`: throws away the trained model and its memory, so the reload really starts from the files.
- `load(MODEL, adapter_path=...)`: a fresh base model with the saved adapter on it. This is exercise 6.

Then PEFT: `PeftModel.from_pretrained` wraps a fresh base model and loads the adapter from the folder.

In [ ]:
from peft import PeftModel


def peft_reply(model):
    ids = tok.apply_chat_template(question, add_generation_prompt=True, return_tensors="pt")
    ids = ids["input_ids"].to("mps")
    out = model.generate(input_ids=ids, max_new_tokens=60, do_sample=False)
    return tok.decode(out[0, ids.shape[1] :], skip_special_tokens=True).strip()


trained_reply = peft_reply(peft_model)
del peft_model, base
torch.mps.empty_cache()

fresh = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("mps")
fresh.generation_config.repetition_penalty = 1.0
loaded = PeftModel.from_pretrained(fresh, peft_dir)
print("trained: ", trained_reply)
print("reloaded:", peft_reply(loaded))

**What each line does**

- `peft_reply(model)`: a greedy reply to the same question (`do_sample=False`), decoded without the prompt (M2).
- `del peft_model, base` and `torch.mps.empty_cache()`: throws the trained model away.
- `PeftModel.from_pretrained(fresh, peft_dir)`: the fresh base model, wrapped, with the saved A and B loaded. This is exercise 5.

Both come back exactly as they were trained. (After only 20 steps the replies are already plainer than the jargon, but `train.py` trains for 100.) This is what makes adapters practical: train once, keep a few megabytes, load them onto the base whenever you need that task.

When you want one plain model instead, to ship or to serve as fast as possible, you merge, as you did in M6. Both tools do it for you: `mlx_lm.fuse` on the command line, and `merge_and_unload()` in PEFT.

## 9. Your task

You've seen every piece. Time to write them yourself.

1. Open `exercises.py`. There are six exercises, each with plain words and numbered steps, and each points to the section above that shows what you need:
   1. `to_chat_record` (section 2)
   2. `mlx_lora_config` (section 3)
   3. `peft_lora_config` (section 4)
   4. `tokens_per_sec` (section 6)
   5. `load_peft_adapter` (section 8)
   6. `load_mlx_adapter` (section 8)
2. Check your work as you go: `make check M=07`. The run tests fail until step 4.
3. Answer the six questions in `answers.toml`. Predict first, then check yourself in this notebook.
4. Train (close this notebook first, or restart its kernel, to free its memory):
   - `uv run python modules/m07_lora_tools/train.py`

   It trains the same adapter with mlx-lm and then with PEFT, each in its own process, on the same batches, for 100 steps (2 epochs). It saves each adapter, loads it back with your exercises 5 and 6, and prints the two runs side by side. About 8 minutes. To pass, each tool's held-out loss must reach 0.8 or lower, its rewrites must score at least 3 chrF points better than the untuned model's, the reloaded adapter must give the same reply for at least 95% of the 60 fixed prompts, and each run must peak under 18 GB.
5. Run `make check M=07` again. When everything passes, M7 turns green on the dashboard.

Stuck? `make hint M=07 EX=3` gives a nudge for exercise 3. Every term is in the glossary in `README.md`.

**Stretch:** run mlx-lm's own command line, `mlx_lm.lora`, on the files `train.py` wrote in `outputs/m07/data/`, with the config from your exercise 2 (the README shows how). It sorts batches by length: is its tokens/sec higher or lower than `train.py`'s, and why?